In [ ]:
#lets install libraries
import operator
from pydantic import BaseModel, Field
from typing import TypedDict, List, Annotated
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langgraph.types import Send
from langchain_google_genai import ChatGoogleGenerativeAI
import os
load_dotenv()

In [ ]:
print("hello world")

In [ ]:
class Task(BaseModel):
    id: int
    title: str
    description: str = Field(..., description="waht to cover")

In [ ]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [ ]:
class State(TypedDict):
    topic: str
    plan: Plan
    sections : Annotated[List[str],operator.add]
    final:str

In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=Plan,
)

In [ ]:
#first Node

print("Orchestrator: Generating plan for topic:")
def Ochestrator(state: State) -> dict:
    # Step 1: Generate a plan for the blog post
    system_msg = SystemMessage("Create a blog plan with 5-7 sections on the following topic.")
    human_msg = HumanMessage(f"Topic: {state['topic']}")

    messages = [system_msg, human_msg]
    plan  = agent.invoke(messages)
    return {"plan": plan}


In [ ]:
def fanout(state: State):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks]

In [ ]:
def Worker (payload: dict) -> dict:
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]
    blog_title = plan.blog_title


    system_msg = SystemMessage("You are a helpful assistant that writes blog sections.")
    human_msg = HumanMessage(
        f"Blog: {blog_title}\n"
        f"Topic: {topic}\n\n"
        f"Section: {task.title}\n"
        f"Brief: {task.brief}\n\n"
        "Return only the section content in Markdown.")

    messages = [system_msg, human_msg]
    section_content = agent.invoke(messages)
    return {"section": [section_content]}

In [ ]:
from pathlib import Path

def Reducer(state: State) -> dict:
    
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{body}\n"

    # ---- save to file ----
    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md}

In [ ]:
g = StateGraph(State)
g.add_node("orchestrator", Ochestrator)
g.add_node("worker", Worker)
g.add_node("reducer", Reducer)

In [ ]:
g.add_edge(START, "Ochestrator")
g.add_conditional_edges("Ochestrator", fanout, "worker")
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()

app

In [ ]:
out = app.invoke({"topic": "Write a blog on Self Attention", "sections": []})